# Basis Pursuit (BP)
Conceito:
O Basis Pursuit relaxa o problema original substituindo a norma ℓ0ℓ0​ pela norma ℓ1ℓ1​, resultando em um problema convexo:
min⁡x∥x∥1sujeito aAx=y.
xmin​∥x∥1​sujeito aAx=y.

Essa abordagem é garantida de recuperar o sinal original sob certas condições (como a propriedade de isometria restrita - RIP).

Implementação:
Esse problema pode ser resolvido usando técnicas de programação convexa, por exemplo, com métodos como o método do ponto interior ou utilizando solvers especializados como o CVX, CVXPY ou SPGL1.

In [1]:
import cvxpy as cp
import numpy as np

# Definindo dimensões
m, n = 50, 200  # m < n
np.random.seed(0)

# Matriz de medição aleatória (por exemplo, Gaussian)
A = np.random.randn(m, n)

# Sinal esparso verdadeiro (k elementos não-nulos)
k = 10
x_true = np.zeros(n)
indices = np.random.choice(n, k, replace=False)
x_true[indices] = np.random.randn(k)

# Medições (sem ruído para simplificação)
y = A @ x_true

# Variável de otimização
x = cp.Variable(n)

# Problema de Basis Pursuit
objective = cp.Minimize(cp.norm1(x))
constraints = [A @ x == y]
problem = cp.Problem(objective, constraints)
problem.solve()

print("Sinal reconstruído:", x.value)


Sinal reconstruído: [ 3.40900951e-11 -3.70055523e-10  3.51213926e-11 -1.58601470e-09
  3.81980783e-11 -2.66168478e-11  3.35423466e-10  2.97573081e-09
 -1.41752729e-10 -2.47929749e-10 -2.06752412e-10 -7.34005164e-11
  5.67358399e-11  1.77402175e-10  1.30876723e-09  5.57791072e-10
 -3.81400759e-11 -8.73268354e-11 -3.21406830e-10  5.60893627e-10
 -3.55246043e-11  7.62536940e-11 -1.38420779e-10  4.77450089e-10
 -9.77195500e-12  8.21620459e-01 -5.94227804e-11 -6.75198405e-11
  2.02304968e-10 -2.20615579e-10 -3.74173942e-12 -9.59450363e-11
 -7.71740371e-11  4.29726703e-10  3.27319621e-01  6.15255210e-11
  3.84581368e-11  9.52852161e-10  8.26515567e-11  5.41919457e-11
  4.16388509e-11 -9.68575675e-11 -1.52095902e-10  6.15073202e-01
 -1.83065214e-11  1.99098879e-10  1.50346466e-10 -4.45690013e-10
  1.22153159e-11 -1.96642703e-10 -1.27428438e-10 -9.67842912e-11
  1.89396741e-10  9.32826551e-10 -3.44830947e-10 -7.15029588e-11
 -8.60240215e-11  1.91651564e-10  5.75250034e-01 -1.13402328e-10
  6.8

# Orthogonal Matching Pursuit (OMP)

Conceito:
OMP é um algoritmo "greedy" que constrói, iterativamente, uma aproximação para o sinal esparso. Em cada iteração, o algoritmo seleciona a coluna de AA que é mais correlacionada com o resíduo atual (a diferença entre yy e a aproximação corrente do sinal). Após selecionar o suporte (índices dos coeficientes não-nulos), é feita uma projeção ortogonal para atualizar a estimativa.

## Pseudocódigo


In [ ]:

Entrada: Matriz A, vetor de medições y, número de iterações k (ou critério de parada)
Inicialização: resíduo r = y, suporte S = {}
Para i = 1 até k:
    1. Seleciona o índice j que maximiza |<A_j, r>|, onde A_j é a j-ésima coluna de A.
    2. Adiciona j ao suporte S.
    3. Resolve o problema de mínimos quadrados:
         x_S = arg min || y - A_S x ||_2,
       onde A_S é a submatriz com colunas em S.
    4. Atualiza o resíduo:
         r = y - A_S x_S.
Retorne a solução completa x preenchida com zeros fora do suporte S.

In [2]:
import numpy as np
from numpy.linalg import lstsq

def omp(A, y, k):
    residual = y.copy()
    idx_support = []
    x_est = np.zeros(A.shape[1])
    
    for _ in range(k):
        # Seleciona a coluna com maior correlação com o residual
        correlations = np.abs(A.T @ residual)
        new_idx = np.argmax(correlations)
        if new_idx in idx_support:
            break  # Evita repetição
        idx_support.append(new_idx)
        
        # Resolve o sistema em mínimos quadrados com o suporte atual
        A_subset = A[:, idx_support]
        x_subset, _, _, _ = lstsq(A_subset, y, rcond=None)
        
        # Atualiza o residual
        residual = y - A_subset @ x_subset
        
        # Se o residual estiver abaixo de um limiar, encerra
        if np.linalg.norm(residual) < 1e-6:
            break

    # Preenche a solução completa
    x_est[idx_support] = x_subset
    return x_est

# Exemplo de uso:
m, n, k = 50, 200, 10
np.random.seed(0)
A = np.random.randn(m, n)
x_true = np.zeros(n)
indices = np.random.choice(n, k, replace=False)
x_true[indices] = np.random.randn(k)
y = A @ x_true

x_rec = omp(A, y, k)
print("Sinal verdadeiro:", x_true)
print("Sinal reconstruído (OMP):", x_rec)
    

Sinal verdadeiro: [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.82162046  0.          0.          0.          0.
  0.          0.          0.          0.          0.32731963  0.
  0.          0.          0.          0.          0.          0.
  0.          0.6150732   0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.57525004  0.
  0.          0.          0.          0.89448959  0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.         -1.39721956
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.

# LASSO (Least Absolute Shrinkage and Selection Operator)

Conceito:
O LASSO é uma técnica que combina um termo de minimização do erro de aproximação com uma penalização ℓ1ℓ1​ para incentivar a esparsidade:
min⁡x12∥Ax−y∥22+λ∥x∥1.
xmin​21​∥Ax−y∥22​+λ∥x∥1​.

O parâmetro λλ controla o trade-off entre a fidelidade à medição e a esparsidade da solução. Uma vantagem do LASSO é que ele lida bem com ruído nas medições.

In [3]:
import numpy as np
from sklearn.linear_model import Lasso

# Definindo dimensões
m, n, k = 50, 200, 10
np.random.seed(0)
A = np.random.randn(m, n)
x_true = np.zeros(n)
indices = np.random.choice(n, k, replace=False)
x_true[indices] = np.random.randn(k)
y = A @ x_true

# Escolha do parâmetro de regularização (pode ser ajustado)
lambda_reg = 0.1

lasso = Lasso(alpha=lambda_reg, fit_intercept=False, max_iter=10000)
lasso.fit(A, y)
x_rec = lasso.coef_

print("Sinal reconstruído (LASSO):", x_rec)


Sinal reconstruído (LASSO): [-0.         -0.          0.         -0.          0.         -0.
  0.          0.04198077 -0.         -0.         -0.          0.
  0.          0.          0.          0.          0.         -0.
 -0.          0.         -0.          0.         -0.          0.
  0.          0.80325411 -0.         -0.          0.         -0.
 -0.          0.         -0.          0.          0.09779038  0.
  0.          0.00231614  0.          0.         -0.         -0.
 -0.          0.52475834 -0.          0.          0.         -0.
 -0.         -0.         -0.         -0.          0.          0.
 -0.         -0.         -0.          0.          0.285934   -0.
  0.          0.          0.          0.60909063 -0.          0.
  0.          0.         -0.         -0.         -0.          0.
  0.          0.          0.         -0.          0.01368436 -1.17687803
 -0.         -0.          0.         -0.         -0.         -0.
 -0.         -0.         -0.          0.         -0.  

# Iterative Hard Thresholding (IHT)

Conceito:
O IHT é um método iterativo que combina um passo de descida do gradiente no problema de mínimos quadrados com uma projeção (thresholding) para garantir esparsidade. Dado um passo de iteração:
x(t+1)=Hk(x(t)+μAT(y−Ax(t))),
x(t+1)=Hk​(x(t)+μAT(y−Ax(t))),

onde Hk(⋅)Hk​(⋅) é um operador que mantém apenas os kk maiores (em módulo) elementos e define os demais como zero, e μμ é um passo de iteração (passo de learning rate).

In [9]:
def hard_thresholding(z, k):
    """Mantém os k maiores elementos (em módulo) de z e zera o restante."""
    idx = np.argsort(np.abs(z))[-k:]
    x_new = np.zeros_like(z)
    x_new[idx] = z[idx]
    return x_new

def iht(A, y, k, mu=0.01, T=1000):
    x_est = np.zeros(A.shape[1])
    for _ in range(T):
        gradient = A.T @ (y - A @ x_est)
        x_est = x_est + mu * gradient
        x_est = hard_thresholding(x_est, k)
    return x_est

# Exemplo de uso:
m, n, k = 50, 200, 10
np.random.seed(0)
A = np.random.randn(m, n)
x_true = np.zeros(n)
indices = np.random.choice(n, k, replace=False)
x_true[indices] = np.random.randn(k)
y = A @ x_true

x_rec_iht = iht(A, y, k, mu=0.001, T=5000)
print("Sinal reconstruído (IHT):", x_rec_iht)


Sinal reconstruído (IHT): [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.74153296  0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.60289196  0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.5078804   0.
  0.          0.          0.          0.81195389  0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.         -1.41273876
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.    